In [1]:
import torch
import itertools
import os
import pyro
import random
import numpy as np
import pyro.distributions as dist
from pyro.infer import Trace_ELBO


from pathlib import Path
from einops import repeat

from pyro_cases.utils.base_vae import BaseVAEwRegister
from pyro_cases.utils.vae_dict import vae_dict
from pyro_cases.utils.variational_dist import VariationalDist

In [2]:
# seed for elbo computation
seed = 7272
pyro.set_rng_seed(seed)
random.seed(seed)
np.random.seed(seed)

In [3]:
# extract_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_07-28_non_amortized_vae_larger_lr_output")
# favi_or_elbo = "elbo"
# output_file_path = Path("/data/scratch/pduan/new_gcvi_output/gcvi_07-28_non_amortized_vae_larger_lr_test_summary.pt")

In [4]:
extract_result_dir = Path("/data/scratch/pduan/new_gcvi_output/new_gcvi_08-02_deep_set_favi_with_fixed_design_output")
favi_or_elbo = "favi"
output_file_path = Path("/data/scratch/pduan/new_gcvi_output/new_gcvi_08-02_deep_set_favi_with_fixed_design_test_summary.pt")

In [5]:
extract_files = os.listdir(extract_result_dir)

In [6]:
print(f"# files: {len(extract_files)}")

# files: 117


In [7]:
valid_files = []
for cf in extract_files:
    extract_results = torch.load(extract_result_dir / cf, map_location="cpu")
    find_error = any([r[f"{favi_or_elbo}_error"] is not None for r in extract_results])
    if find_error:
        continue
    valid_files.append(cf)

In [8]:
len(valid_files)

117

In [9]:
def get_est_mu_sigma2(variational_dist: VariationalDist, results, tag):
    est_mu = []
    est_sigma2 = []
    for r in results:
        raw_pred = r[f"{tag}_test_result_dict"]["raw_pred"]
        mu, sigma2 = variational_dist.get_mu_sigma2(raw_pred)
        est_mu.append(mu)  # (num_obs, k)
        est_sigma2.append(sigma2)
    est_mu = torch.stack(est_mu, dim=0)  # (r, num_obs, k)
    est_sigma2 = torch.stack(est_sigma2, dim=0)  # (r, num_obs, k)
    return est_mu, est_sigma2

In [10]:
def kl_div_two_normal(p_mu, p_sigma2, q_mu, q_sigma2):
    return torch.log(q_sigma2.sqrt()) - torch.log(p_sigma2.sqrt()) + (p_sigma2 + (p_mu - q_mu) ** 2) / (2 * q_sigma2) - 0.5

In [11]:
def get_kl_for_repeats(est_mu_sigma2: torch.Tensor):
    # est_mu_sigma2: (r, num_obs, k, 2)
    assert est_mu_sigma2.shape[-1] == 2
    combs = torch.tensor(list(itertools.combinations(range(est_mu_sigma2.shape[0]), 2)))  # (c, 2)
    perms = torch.cat([combs, combs.flip(dims=[-1])], dim=0)  # (2c, 2)
    perm_est_mu_sigma2 = est_mu_sigma2[perms]  # (2c, 2, num_obs, k, 2)
    return kl_div_two_normal(p_mu=perm_est_mu_sigma2[:, 0, :, :, 0],
                            p_sigma2=perm_est_mu_sigma2[:, 0, :, :, 1],
                            q_mu=perm_est_mu_sigma2[:, 1, :, :, 0],
                            q_sigma2=perm_est_mu_sigma2[:, 1, :, :, 1])  # (2c, num_obs, k)

In [12]:
def D_measure(p_mu, p_sigma2, q_mu, q_sigma2):
    return torch.abs(p_mu - q_mu) / (torch.sqrt((p_sigma2 + q_sigma2) / 2) + 0.01)

In [13]:
def get_D_measure_for_repeats(est_mu_sigma2: torch.Tensor):
    # est_mu_sigma2: (r, num_obs, k, 2)
    assert est_mu_sigma2.shape[-1] == 2
    combs = torch.tensor(list(itertools.combinations(range(est_mu_sigma2.shape[0]), 2)))  # (c, 2)
    perms = torch.cat([combs, combs.flip(dims=[-1])], dim=0)  # (2c, 2)
    perm_est_mu_sigma2 = est_mu_sigma2[perms]  # (2c, 2, num_obs, k, 2)
    return D_measure(p_mu=perm_est_mu_sigma2[:, 0, :, :, 0],
                            p_sigma2=perm_est_mu_sigma2[:, 0, :, :, 1],
                            q_mu=perm_est_mu_sigma2[:, 1, :, :, 0],
                            q_sigma2=perm_est_mu_sigma2[:, 1, :, :, 1])  # (2c, num_obs, k)

In [14]:
def energy_fn(est_dist, true_value, m=8):
    est_samples = est_dist.sample((m,))  # (m, r, b, k)
    first_term = (est_samples - true_value).abs().mean(dim=0)
    second_term = (est_samples.unsqueeze(1) - est_samples.unsqueeze(0)).abs().mean(dim=(0, 1)) * m / (m - 1)
    return first_term - 0.5 * second_term  # (r, b, k)

In [15]:
def extract_vsbc(results, tag):
    vsbc_list = []
    for r in results:
        vsbc_list.append(r[f"{tag}_vsbc"])  # (k, s)
    return torch.stack(vsbc_list, dim=0)  # (r, k, s)

In [16]:
def wasserstein_distance_to_unif(u: torch.Tensor):
    assert u.ndim == 3  # (r, k, s)
    unif_samples = torch.linspace(0.0, 1.0, u.shape[-1]).view(1, 1, -1)
    sorted_u = torch.sort(u, dim=-1, descending=False)[0]  # (r, k, s)
    return torch.abs(sorted_u - unif_samples).mean(dim=-1)  # (r, k)

In [17]:
def print_value(est_value, tag):
    print(f"mean({tag}): {est_value.mean():.3e}")
    print(f"median({tag}): {est_value.median():.3e}")

In [18]:
output_file_dict = {}
for i, vf in enumerate(valid_files):
    extract_results = torch.load(extract_result_dir / vf, map_location="cpu")

    task_name = extract_results[0]["task"]
    cur_vae: BaseVAEwRegister = vae_dict[task_name](hidden_dim=1, use_neural_network=False)
    cur_vae.do_register(1)

    print("=" * 50)
    print(f"[{i + 1}] task name: {task_name}")

    test_sample_dict = extract_results[0][f"{favi_or_elbo}_test_sample_dict"]
    true_theta = cur_vae.extract_theta(test_sample_dict)
    n_test_obs = true_theta.shape[0]
    
    # test kl
    est_mu, est_sigma2 = get_est_mu_sigma2(cur_vae.variational_dist, extract_results, tag=favi_or_elbo)
    kl_r = get_kl_for_repeats(torch.stack([est_mu, est_sigma2], dim=-1))
    print_value(kl_r, tag="kl")

    # test D
    D_value = get_D_measure_for_repeats(torch.stack([est_mu, est_sigma2], dim=-1))
    print_value(D_value, tag="D")
    
    # test logp
    logp = dist.Normal(est_mu, est_sigma2.sqrt()).log_prob(repeat(true_theta, 
                                                                  "b k -> r b k", 
                                                                  r=len(extract_results)))
    print_value(logp, tag="logp")

    # test elbo
    elbo_value = []
    elbo_error = False
    for sub_favi_est_mu, sub_favi_est_sigma2, sub_r in zip(est_mu, 
                                                            est_sigma2, 
                                                            extract_results, 
                                                            strict=True):
        cur_vae.set_fixed_raw_pred(sub_r[f"{favi_or_elbo}_test_result_dict"]["raw_pred"])
        try:
            elbo_value.append(-1 * Trace_ELBO(num_particles=1).loss(cur_vae.model, 
                                                                    cur_vae.guide, 
                                                                    n_test_obs, 
                                                                    test_sample_dict))
        except Exception as e:
            print(f"get error during elbo compute: {str(e)}")
            elbo_error = True
            break
    if elbo_error:
        continue
    elbo_value = torch.tensor(elbo_value)
    print_value(elbo_value, tag="elbo")

    # test energy
    energy = energy_fn(dist.Normal(est_mu, est_sigma2.sqrt()), true_theta)
    print_value(energy, tag="energy")

    # test vsbc
    vsbc = extract_vsbc(extract_results, tag=favi_or_elbo)
    vsbc_d = wasserstein_distance_to_unif(vsbc)
    print_value(vsbc_d, "vsbc-to-uniform distance")

    output_file_dict[task_name] = {
        "kl": (kl_r.mean().item(), kl_r.median().item()),
        "D": (D_value.mean().item(), D_value.median().item()),
        "logp": (logp.mean().item(), logp.median().item()),
        "elbo": (elbo_value.mean().item(), elbo_value.median().item()),
        "energy": (energy.mean().item(), energy.median().item()),
        "vsbc-to-uniform_dist": (vsbc_d.mean().item(), vsbc_d.median().item()),
    }
torch.save(output_file_dict, output_file_path)

[1] task name: arm_kidiq_interaction_c
mean(kl): 1.410e+00
median(kl): 1.906e-02
mean(D): 2.035e-01
median(D): 1.488e-01
mean(logp): -8.973e-01
median(logp): -1.725e+00
mean(elbo): -1.602e+09
median(elbo): -1.829e+09
mean(energy): 3.983e-01
median(energy): 3.918e-01
mean(vsbc-to-uniform distance): 1.880e-01
median(vsbc-to-uniform distance): 1.926e-01
[2] task name: arm_electric_multi_preds
mean(kl): 7.706e-01
median(kl): 1.366e-02
mean(D): 5.737e-02
median(D): 2.872e-02
mean(logp): 1.171e+00
median(logp): 2.271e+00
mean(elbo): -8.877e+06
median(elbo): -1.298e+07
mean(energy): 1.586e-01
median(energy): 1.262e-02
mean(vsbc-to-uniform distance): 1.165e-01
median(vsbc-to-uniform distance): 1.758e-01
[3] task name: arm_wells_probit
mean(kl): 1.695e-05
median(kl): 8.821e-06
mean(D): 3.517e-03
median(D): 2.817e-03
mean(logp): 2.382e-01
median(logp): 4.942e-01
mean(elbo): -3.306e+04
median(elbo): -3.306e+04
mean(energy): 1.075e-01
median(energy): 7.638e-02
mean(vsbc-to-uniform distance): 1.047